In [1]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.landesdatenbank.nrw.de/
df = pd.read_csv("../data/raw/2024/12411-15i.csv", sep=";", encoding="latin1", skiprows=3)

# Überblick:
print(df.head())
df.info()


   Unnamed: 0  Unnamed: 1                      Unnamed: 2  \
0         NaN         NaN                             NaN   
1  31.12.2024         5.0             Nordrhein-Westfalen   
2  31.12.2024        51.0    Düsseldorf, Regierungsbezirk   
3  31.12.2024      5111.0         Düsseldorf, krfr. Stadt   
4  31.12.2024      5112.0           Duisburg, krfr. Stadt   

  Bevölkerungsstand (Basis Zensus 2022 und früher)  \
0                                           Anzahl   
1                                         18034454   
2                                          5244379   
3                                           618685   
4                                           502270   

  Bodenfläche (Katasterfläche) in qkm Bevölkerungsdichte  
0                                 qkm         EW pro qkm  
1                            34112,44              528,7  
2                             5292,33              990,9  
3                              217,41             2845,7  
4            

In [2]:
#relevante Spalten filtern
df = df[['Unnamed: 2', 'Bevölkerungsstand (Basis Zensus 2022 und früher)', 'Bevölkerungsdichte']]
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df)

                         Unnamed: 2  \
0                               NaN   
1               Nordrhein-Westfalen   
2      Düsseldorf, Regierungsbezirk   
3           Düsseldorf, krfr. Stadt   
4             Duisburg, krfr. Stadt   
..                              ...   
440                    Lünen, Stadt   
441                 Schwerte, Stadt   
442                     Selm, Stadt   
443                     Unna, Stadt   
444                    Werne, Stadt   

    Bevölkerungsstand (Basis Zensus 2022 und früher) Bevölkerungsdichte  
0                                             Anzahl         EW pro qkm  
1                                           18034454              528,7  
2                                            5244379              990,9  
3                                             618685             2845,7  
4                                             502270             2157,1  
..                                               ...                ...  
440          

In [3]:
# Spalten umbenennen
df = df.rename(columns={"Unnamed: 2": "Name"})

# Ints und Strings anpassen
df["Name"] = df["Name"].str.strip()

df["Bevölkerungsdichte"] = (
    df["Bevölkerungsdichte"]
    .str.replace(",", ".", regex=False)
    .apply(pd.to_numeric, errors="coerce")
    .astype(float)
)

df["Bevölkerungsstand (Basis Zensus 2022 und früher)"] = df["Bevölkerungsstand (Basis Zensus 2022 und früher)"].apply(pd.to_numeric, errors="coerce").astype("Int64")


# nach Kreisen und kreisfreien Städten filtern 
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]

In [4]:
print(df.head())
print(df[df["Bevölkerungsdichte"].isna()])

                           Name  \
3       Düsseldorf, krfr. Stadt   
4         Duisburg, krfr. Stadt   
5            Essen, krfr. Stadt   
6          Krefeld, krfr. Stadt   
7  Mönchengladbach, krfr. Stadt   

   Bevölkerungsstand (Basis Zensus 2022 und früher)  Bevölkerungsdichte  
3                                            618685              2845.7  
4                                            502270              2157.1  
5                                            574682              2732.2  
6                                            231406              1679.5  
7                                            267213              1567.5  
                   Name  Bevölkerungsstand (Basis Zensus 2022 und früher)  \
75  Aachen, krfr. Stadt                                              <NA>   
90        Aachen, Kreis                                              <NA>   

    Bevölkerungsdichte  
75                 NaN  
90                 NaN  


In [5]:
# Zeile löschen und Name anpassen
# df = df[df["Name"] != "Essen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, Kreis"]
df = df[df["Name"] != "Aachen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, krfr. Stadt (ab 21.10.2009)"]


df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53 entries, 3 to 434
Data columns (total 3 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Name                                              53 non-null     object 
 1   Bevölkerungsstand (Basis Zensus 2022 und früher)  53 non-null     Int64  
 2   Bevölkerungsdichte                                53 non-null     float64
dtypes: Int64(1), float64(1), object(1)
memory usage: 1.7+ KB


In [6]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df


df = clean_and_sort(df,"Bevölkerungsdichte")

# Namen bereinigen
validate_df(df,
    not_null=["Bevölkerungsdichte"],
    positive=["Bevölkerungsdichte"],
    numeric=["Bevölkerungsdichte"],
    key_cols=["Name"])


DataFrame: alle Checks bestanden


[]

In [7]:
# speichern
df.to_csv("../data/processed/bevoelkerungsdichte_2024.csv", index=False)